In [ ]:
import json
from pathlib import Path

import h5py
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# 0) 用户配置：数据集目录（硬编码，与你脚本一致）
# ============================================================
DATA_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal")
OUT_DIR  = DATA_DIR.parent / "qc_analysis_results_batch" / "thesis_figures" / "chapter4" / "fig4_1_curtain"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_MODALITY = 341   # 0-based, MPRAGE
MIN_BRAIN_RATIO    = 0.05  # 与你现有脚本一致的slice有效性阈值
MAX_PIXELS         = 50_000
RANDOM_SEED        = 42
SORT_BY_MPRAGE     = True  # True: sort columns by MPRAGE intensity (recommended)

# 家族边界（0-based，与 multimodal_qc_tasks_extra.py 一致）
GROUPS_FOR_BOUNDS = [
    ("QTI",                  0,  14),
    ("b_lin",               15,  95),
    ("b_plan",              96, 176),
    ("b_spher",            177, 224),
    ("CEST",               225, 228),
    ("M0",                 229, 229),
    ("z spectrum low B1",  230, 283),
    ("M0",                 284, 285),
    ("z spectrum high B1", 286, 339),
    ("M0",                 340, 340),
    ("MPRAGE",             341, 341),
    ("QSM_TE",             342, 346),
    ("TE_avg",             347, 347),
    ("SMWI_on_avg",        348, 348),
    ("SMWI_on_first",      349, 349),
    ("QSM",                350, 350),
]

# ============================================================
# 1) Thesis-style matplotlib（德国论文常见：小字号、嵌入字体、简洁）
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.unicode_minus": False,
    "pdf.fonttype": 42,  # embed TrueType in PDF
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# ============================================================
# 2) 工具函数：选 slice、读取 slice（只读需要的 slice，避免整4D加载）
# ============================================================
def choose_slice_max_mask(mask3d: np.ndarray, axis: int, min_ratio: float) -> tuple[int, float]:
    # mask3d shape: (X,Y,Z)
    if axis == 0:   # sagittal: vary X, average over (Y,Z)
        ratios = mask3d.mean(axis=(1, 2))
    elif axis == 1: # coronal: vary Y, average over (X,Z)
        ratios = mask3d.mean(axis=(0, 2))
    elif axis == 2: # axial: vary Z, average over (X,Y)
        ratios = mask3d.mean(axis=(0, 1))
    else:
        raise ValueError("axis must be 0/1/2")

    valid = np.where(ratios >= min_ratio)[0]
    if valid.size == 0:
        k = int(np.argmax(ratios))
        return k, float(ratios[k])
    k = int(valid[np.argmax(ratios[valid])])
    return k, float(ratios[k])

def read_slice_all_channels(h5f: h5py.File, axis: int, slice_idx: int) -> np.ndarray:
    """
    Returns slice array with shape (H, W, C=351)
    Supports storage as (351, X, Y, Z) or (X, Y, Z, 351).
    """
    dset = h5f["data"]
    shp = dset.shape

    if len(shp) != 4:
        raise ValueError(f"Unexpected data shape: {shp}")

    if shp[0] == 351:
        # stored as (C, X, Y, Z)
        if axis == 0:   # sagittal (X fixed) -> (C, Y, Z) -> (Y, Z, C)
            arr = np.asarray(dset[:, slice_idx, :, :])
            arr = np.moveaxis(arr, 0, -1)
        elif axis == 1: # coronal (Y fixed)  -> (C, X, Z) -> (X, Z, C)
            arr = np.asarray(dset[:, :, slice_idx, :])
            arr = np.moveaxis(arr, 0, -1)
        else:           # axial (Z fixed)    -> (C, X, Y) -> (X, Y, C)
            arr = np.asarray(dset[:, :, :, slice_idx])
            arr = np.moveaxis(arr, 0, -1)
    elif shp[-1] == 351:
        # stored as (X, Y, Z, C)
        if axis == 0:
            arr = np.asarray(dset[slice_idx, :, :, :])  # (Y,Z,C)
        elif axis == 1:
            arr = np.asarray(dset[:, slice_idx, :, :])  # (X,Z,C)
        else:
            arr = np.asarray(dset[:, :, slice_idx, :])  # (X,Y,C)
    else:
        raise ValueError(f"Cannot infer channel axis from shape: {shp}")

    if arr.shape[-1] != 351:
        raise ValueError(f"Slice last dim must be 351, got {arr.shape}")
    return arr.astype(np.float32, copy=False)

def read_mask_slice(h5f: h5py.File, axis: int, slice_idx: int) -> np.ndarray:
    m = h5f["region_mask"]
    if axis == 0:
        return np.asarray(m[slice_idx, :, :]).astype(bool)
    elif axis == 1:
        return np.asarray(m[:, slice_idx, :]).astype(bool)
    else:
        return np.asarray(m[:, :, slice_idx]).astype(bool)

# ============================================================
# 3) Curtain：对一个 2D slice 做 351×N 像素
# ============================================================
def build_curtain_u8(slice_hwc: np.ndarray,
                     mask2d: np.ndarray,
                     ref_idx: int = REFERENCE_MODALITY,
                     max_pixels: int = MAX_PIXELS,
                     sort_by_mprage: bool = True,
                     seed: int = RANDOM_SEED) -> tuple[np.ndarray, dict]:
    H, W, C = slice_hwc.shape
    assert C == 351

    idx_all = np.flatnonzero(mask2d.ravel())
    if idx_all.size == 0:
        raise ValueError("Mask slice is empty; cannot build curtain.")

    rng = np.random.default_rng(seed)
    idx = idx_all
    if idx_all.size > max_pixels:
        idx = rng.choice(idx_all, size=max_pixels, replace=False)

    V = slice_hwc.reshape(-1, C)[idx, :]  # (N,351)

    if sort_by_mprage:
        order = np.argsort(V[:, ref_idx])
        V = V[order, :]

    # per-modality 1–99% normalization (vectorized)
    p = np.percentile(V, [1, 99], axis=0)  # shape (2,351)
    denom = (p[1] - p[0]) + 1e-8
    VN = np.clip((V - p[0]) / denom, 0.0, 1.0)

    curtain_u8 = (VN * 255.0).astype(np.uint8).T  # (351, N)

    meta = {
        "H": int(H), "W": int(W), "C": int(C),
        "n_pixels_total": int(idx_all.size),
        "n_pixels_used": int(V.shape[0]),
        "sort_by_mprage": bool(sort_by_mprage),
        "ref_modality_idx": int(ref_idx),
        "seed": int(seed),
        "max_pixels": int(max_pixels),
        "norm": "per-modality percentile (1,99) within selected slice pixels"
    }
    return curtain_u8, meta

def plot_save_curtain(curtain_u8: np.ndarray,
                      axis_name: str,
                      slice_idx: int,
                      subject_id: str,
                      out_dir: Path,
                      meta: dict,
                      dpi_png: int = 600):
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    im = ax.imshow(
        curtain_u8,
        cmap="gray",
        aspect="auto",
        interpolation="nearest",
        rasterized=True  # keep PDF size reasonable
    )

    # Group labels (readable, thesis-friendly)
    mids = [(a + b) / 2 for (_, a, b) in GROUPS_FOR_BOUNDS]
    names = [n for (n, _, _) in GROUPS_FOR_BOUNDS]
    ax.set_yticks(mids)
    ax.set_yticklabels(names)

    # Subtle boundary lines
    for _, _, b in GROUPS_FOR_BOUNDS:
        ax.axhline(b + 0.5, lw=0.4, color="0.75")

    ax.set_xticks([])
    ax.set_xlabel(f"ROI pixels in {axis_name} slice {slice_idx} (n={meta['n_pixels_used']:,}; "
                  f"{'sorted by MPRAGE' if meta['sort_by_mprage'] else 'random order'})")
    ax.set_ylabel("Modality families (351 channels)")

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Normalized intensity (1–99% per modality)")

    fig.tight_layout()

    base = out_dir / f"fig4_1_curtain_{subject_id}_{axis_name}_slice{slice_idx:03d}"
    fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.02)
    fig.savefig(base.with_suffix(".png"), dpi=dpi_png, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)

    return {
        "axis": axis_name,
        "slice_idx": int(slice_idx),
        "pdf": str(base.with_suffix(".pdf")),
        "png": str(base.with_suffix(".png")),
    }

# ============================================================
# 4) 主流程：默认第一个 patient，三轴各一张
# ============================================================
mat_files = sorted(DATA_DIR.glob("*.mat"))
if len(mat_files) == 0:
    raise FileNotFoundError(f"No .mat files found in: {DATA_DIR}")

mat_path = mat_files[0]
subject_id = mat_path.stem
print(f"[INFO] Using subject: {subject_id}")
print(f"[INFO] Source: {mat_path}")

# 只把 mask3d 读入内存（体积约 384*336*256，OK），用于选切片
with h5py.File(mat_path, "r") as f:
    mask3d = np.asarray(f["region_mask"][:]).astype(bool)

axes = [("sagittal", 0), ("coronal", 1), ("axial", 2)]
all_meta = {"subject": subject_id, "mat_path": str(mat_path), "figures": []}

with h5py.File(mat_path, "r") as f:
    for axis_name, axis in axes:
        slice_idx, mask_ratio = choose_slice_max_mask(mask3d, axis=axis, min_ratio=MIN_BRAIN_RATIO)
        mask2d = read_mask_slice(f, axis=axis, slice_idx=slice_idx)
        slice_hwc = read_slice_all_channels(f, axis=axis, slice_idx=slice_idx)

        curtain_u8, meta = build_curtain_u8(
            slice_hwc=slice_hwc,
            mask2d=mask2d,
            ref_idx=REFERENCE_MODALITY,
            max_pixels=MAX_PIXELS,
            sort_by_mprage=SORT_BY_MPRAGE,
            seed=RANDOM_SEED
        )
        meta.update({"axis": axis_name, "axis_idx": axis, "slice_idx": int(slice_idx), "mask_ratio": float(mask_ratio)})

        out_info = plot_save_curtain(
            curtain_u8=curtain_u8,
            axis_name=axis_name,
            slice_idx=slice_idx,
            subject_id=subject_id,
            out_dir=OUT_DIR,
            meta=meta,
            dpi_png=600
        )
        all_meta["figures"].append({**meta, **out_info})
        print(f"[OK] Saved {axis_name}: slice={slice_idx}, mask_ratio={mask_ratio:.3f}")

meta_path = OUT_DIR / f"fig4_1_curtain_{subject_id}_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(all_meta, f, indent=2, ensure_ascii=False)

print(f"[OK] Meta: {meta_path}")
print(f"[OK] Output dir: {OUT_DIR}")
